# 3D-VP-LCP Tutorial

This notebook walks through the full **3-D Vertical Permeability Least-Cost Path** pipeline
using a small synthetic LiDAR dataset.

## Pipeline steps
1. Load LiDAR data
2. Normalize heights
3. Voxelize the point cloud
4. Compute vertical gap fraction (VGF)
5. Build a resistance surface
6. Apply species-specific morphology filter
7. Construct a 3-D graph and run least-cost-path routing
8. Visualize the corridor

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))



import numpy as np

from vp_lcp.lidar_io import load_laz, normalize_heights

from vp_lcp.voxelizer import voxelize

from vp_lcp.vertical_gap import vertical_gap_fraction

from vp_lcp.resistance import compute_resistance

from vp_lcp.species_filter import apply_species_filter

from vp_lcp.graph3d import build_graph, least_cost_path

from vp_lcp.visualization import corridor_to_points, resistance_to_2d

## 1. Load and normalize the sample LiDAR

In [ ]:
pts = load_laz('../data/sample_lidar.las')
pts = normalize_heights(pts, ground_percentile=2.0)
# Keep only above-ground points
pts = pts[pts[:, 2] > 0.5]
print(f'{len(pts):,} points loaded')

## 2. Voxelize

In [ ]:
vox_size = 2.0  # metres
grid = voxelize(pts, vox_size)
print(f'{len(grid.counts):,} occupied voxels')

## 3. Vertical gap fraction

In [ ]:
vgf = vertical_gap_fraction(grid, tau=0.2)
print(f'VGF computed for {len(vgf):,} voxels')
print(f'Mean VGF: {np.mean(list(vgf.values())):.3f}')

## 4. Build resistance surface

In [ ]:
R = compute_resistance(grid, vgf, alpha=1.2, beta=0.8)
print(f'Resistance computed for {len(R):,} voxels')
print(f'Range: [{min(R.values()):.2f}, {max(R.values()):.2f}]')

## 5. Species filter

Example: a mid-story mammal that moves at 2-8 m above ground,
needs 1.5 m of vertical clearance, and requires a corridor at
least 2 voxels wide.

In [ ]:
R_filt = apply_species_filter(
    R, grid, vgf,
    h_min=2.0, h_max=8.0,
    h_clear=1.5, w_clear=2,
    vgf_thresh=0.6,
)
print(f'{len(R_filt):,} voxels survive species filtering (from {len(R):,})')

## 6. Graph construction and least-cost path

In [ ]:
G = build_graph(R_filt, vox_size, neighbours=6)
print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Pick source/target as min-x and max-x voxels
keys = list(R_filt)
if keys:
    keys_arr = np.array(keys)
    source_vox = [keys[int(keys_arr[:, 0].argmin())]]
    target_vox = [keys[int(keys_arr[:, 0].argmax())]]
    path = least_cost_path(G, source_vox, target_vox)
    print(f'Least-cost path: {len(path)} voxels')
else:
    print('No passable voxels — try relaxing filter parameters.')
    path = []

## 7. Visualize

In [ ]:
import matplotlib.pyplot as plt

surface = resistance_to_2d(R_filt, grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2-D cost surface
im = axes[0].imshow(surface.T, origin='lower', cmap='viridis_r')
fig.colorbar(im, ax=axes[0], label='Min resistance')
axes[0].set_title('2-D Cost Surface')
axes[0].set_xlabel('X voxel index')
axes[0].set_ylabel('Y voxel index')

# 3-D corridor projected to X-Z
if path:
    coords = corridor_to_points(path, grid)
    axes[1].scatter(coords[:, 0], coords[:, 2], c=np.arange(len(coords)), cmap='cool', s=20)
    axes[1].set_title('Corridor Profile (X vs Height)')
    axes[1].set_xlabel('X (m)')
    axes[1].set_ylabel('Height (m)')
else:
    axes[1].text(0.5, 0.5, 'No path found', transform=axes[1].transAxes, ha='center')

plt.tight_layout()
plt.show()